In [1]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [2]:
input_data = "input_files/mathematical_reasoning_0015.csv"
output_data = "example_output_files/mathematical_reasoning_0015.xlsx"

In [ ]:
## Open AI API
os.environ["OPENAI_API_KEY"] = "your openai api key here"

## common functions

In [4]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [5]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [21]:
"""
    This function is different a bit among different tasks, here is for the mathematical reasoning task
"""
def construct_annotation(df, results, task_name="semantic_matching"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []

    
    for res in results:
        # segmentation_name = res.get("segmentation_name")
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"second_round_filter/segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)
        # potential_seg_names = []

        # ## 
        # for seg_name in real_segmentation_names:
        #     normalized_segmentation_name = segmentation_name.lower()
        #     normalized_seg_name = seg_name.lower()
        #     if (normalized_segmentation_name in normalized_seg_name) or (normalized_seg_name in normalized_segmentation_name):
        #         potential_seg_names.append(seg_name)
        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        
        ##special for mathematical reasoning task
        answers = {"extracted_value": res["error_tags"].get("reported_value"), "calculated_value": res["error_tags"].get("correct_value"), "is_correct": "Yes"}
        special_inputs = {"tag": res["error_tags"].get("tag"), "period": res["error_tags"].get("period")}

        for segmentation_name_ in real_segmentation_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True
            ## check the instance document
            check_tag = special_inputs.get("tag")
            
            try:
                if check_tag in temp_ins:
                    pass
                elif check_tag.replace(":","") in temp_ins:
                    pass
                elif check_tag.replace(":","_") in temp_ins:
                    pass
                else:
                    is_appear = False
               
                if is_appear:
                    parse = True
                    cal = temp_cal
                    pre = temp_pre
                    defn = temp_defn
                    lab = temp_lab
                    ins = temp_ins
                    sch = temp_sch
                    seg_path = os.path.join(folder_name, segmentation_name_)
                    break
            except Exception as e:
                print(f"Errors {e}")
            
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(special_inputs)
        outputByLLM.append(res)
        
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for relationship extraction task

In [25]:
mr_15_df = pd.read_csv(input_data)

In [9]:
mr_15_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0015.1964,"2022-01-14 17:30:24,842 [DQC.US.0015.1964] us-...",2022-01-14,ERROR,DQC,15,1883835,ESG Inc.,2022-01-14 16:59:00,0001520138-22-000017,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_05E_edei%2D%2DC...,10-Q,http://www.sec.gov/Archives/edgar/data/1883835...,2022-01-14,2022,412260,https://www.sec.gov/Archives/edgar/data/188383...,2836,10q-plasma-20211130
1,DQC.US.0015.1307,"2022-02-10 17:03:12,659 [DQC.US.0015.1307] us-...",2022-02-10,ERROR,DQC,15,1070412,CNX Resources Corporation,2022-02-10 16:20:00,0001070412-22-000011,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/1070412...,2022-02-10,2022,417713,https://www.sec.gov/Archives/edgar/data/107041...,1311,10k-cnx-20211231
2,DQC.US.0015.757,"2022-02-17 21:02:49,520 [DQC.US.0015.757] us-g...",2022-02-18,ERROR,DQC,15,23194,"COMSTOCK RESOURCES, INC.",2022-02-17 15:41:00,0000023194-22-000011,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/23194/0...,2022-02-17,2022,420682,https://www.sec.gov/Archives/edgar/data/23194/...,1311,10k-crk-20211231
3,DQC.US.0015.576,"2022-02-23 06:12:12,069 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,1841666,APA Corp,2022-02-22 14:30:00,0001784031-22-000009,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/1841666...,2022-02-22,2022,435265,https://www.sec.gov/Archives/edgar/data/184166...,1311,10k-apa-20211231
4,DQC.US.0015.576,"2022-02-23 06:12:12,070 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,1841666,APA Corp,2022-02-22 14:30:00,0001784031-22-000009,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/1841666...,2022-02-22,2022,435265,https://www.sec.gov/Archives/edgar/data/184166...,1311,10k-apa-20211231
5,DQC.US.0015.576,"2022-02-23 06:12:59,857 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,6769,APACHE CORPORATION,2022-02-22 14:34:00,0001784031-22-000010,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/6769/00...,2022-02-22,2022,435268,https://www.sec.gov/Archives/edgar/data/6769/0...,1311,10k-apa-20211231
6,DQC.US.0015.576,"2022-02-23 06:12:59,858 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,6769,APACHE CORPORATION,2022-02-22 14:34:00,0001784031-22-000010,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/6769/00...,2022-02-22,2022,435268,https://www.sec.gov/Archives/edgar/data/6769/0...,1311,10k-apa-20211231
7,DQC.US.0015.579,"2022-12-29 16:45:43,438 [DQC.US.0015.579] us-g...",2022-12-29,ERROR,DQC,15,10048,"BARNWELL INDUSTRIES, INC.",2022-12-29 16:12:00,0000010048-22-000029,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/10048/0...,2022-12-29,2022,537848,https://www.sec.gov/Archives/edgar/data/10048/...,1311,10k-brn-20220930
8,DQC.US.0015.7166,"2022-12-29 16:45:43,437 [DQC.US.0015.7166] us-...",2022-12-29,ERROR,DQC,15,10048,"BARNWELL INDUSTRIES, INC.",2022-12-29 16:12:00,0000010048-22-000029,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/10048/0...,2022-12-29,2022,537848,https://www.sec.gov/Archives/edgar/data/10048/...,1311,10k-brn-20220930
9,DQC.US.0015.7635,"2022-12-29 16:45:43,439 [DQC.US.0015.7635] srt...",2022-12-29,ERROR,DQC,15,10048,"BARNWELL INDUSTRIES, INC.",2022-12-29 16:12:00,0000010048-22-000029,US GAAP 2021,XBRL Document Created with Wdesk from Workiva,10-K,http://www.sec.gov/Archives/edgar/data/10048/0...,2022-12-29,2022,537848,https://w

## LLM judge assisstant

In [14]:
Client = OpenAI()

In [15]:
mr_15_system_prompt = """You are given a DQC validation message. Your task is to extract the following key information:

1. **Erroneous Tag Information:**:
   Identify the primary us-gaap tag in the message that is reported with an incorrect value due to sign (positive/negative) error. For this tag, extract:
   
   - `tag`: The us-gaap element tag with the error.
   - `period`: The reporting period for this tag, as shown in the "Period" field.
   - `reported_value`: The actual value reported in the filing, as stated in the message or "Value" field.
   - `correct_value`: The correct value, which should be the absolute value of the reported value (i.e., reported_value without the negative sign), as implied or explicitly required by the message.
   - `reason`: Briefly explain why the reported value is considered incorrect, based on the message content (for example, "This element should not have a negative value. The amount should be input as a positive value and, if necessary, a negated label should be provided.").
   
Return your result as structured JSON in the following format:
```json
{
  "error_tags": 
    {
      "tag": "...",
      "period": "...",
      "reported_value": "...",
      "correct_value": "...",
      "reason": "..."
    }
}

"""

In [16]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": mr_15_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [17]:
res = []
for _, row in tqdm(mr_15_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## record the folder name
    response["folder_name"] = folder_name
    res.append(response)

10it [00:29,  2.94s/it]


In [20]:
res[8]

{'error_tags': {'tag': 'us-gaap:CostsIncurredDevelopmentCosts',
  'period': '2021-10-01 to 2022-09-30',
  'reported_value': '-121,000',
  'correct_value': '121,000',
  'reason': 'This element should not have a negative value. The amount should be input as a positive value and, if necessary, a negated label should be provided.'},
 'folder_name': '10k-brn-20220930'}

In [22]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
"""
new_mr_15_df = construct_annotation(mr_15_df, res, task_name="mathematical_reasoning")

In [23]:
new_mr_15_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0015.1964,"2022-01-14 17:30:24,842 [DQC.US.0015.1964] us-...",2022-01-14,ERROR,DQC,15,1883835,ESG Inc.,2022-01-14 16:59:00,0001520138-22-000017,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:PaymentsToAcquirePropertyPlan...,"{'extracted_value': '-1284', 'calculated_value...",10q-plasma-20211130/CondensedStatementOfCashFlows,True,{'error_tags': {'tag': 'us-gaap:PaymentsToAcqu...
1,DQC.US.0015.1307,"2022-02-10 17:03:12,659 [DQC.US.0015.1307] us-...",2022-02-10,ERROR,DQC,15,1070412,CNX Resources Corporation,2022-02-10 16:20:00,0001070412-22-000011,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:IncomeTaxReconciliationTaxCre...,"{'extracted_value': '-2,881,000', 'calculated_...",10k-cnx-20211231/IncomeTaxesEffectiveTaxRateRe...,True,{'error_tags': {'tag': 'us-gaap:IncomeTaxRecon...
2,DQC.US.0015.757,"2022-02-17 21:02:49,520 [DQC.US.0015.757] us-g...",2022-02-18,ERROR,DQC,15,23194,"COMSTOCK RESOURCES, INC.",2022-02-17 15:41:00,0000023194-22-000011,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:DerivativeAverageForwardPrice...,"{'extracted_value': '-0.16', 'calculated_value...",10k-crk-20211231/DerivativeFinancialInstrument...,True,{'error_tags': {'tag': 'us-gaap:DerivativeAver...
3,DQC.US.0015.576,"2022-02-23 06:12:12,069 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,1841666,APA Corp,2022-02-22 14:30:00,0001784031-22-000009,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:CostsIncurredAcquisitionOfOil...,"{'extracted_value': '-157,000,000', 'calculate...",10k-apa-20211231/SUPPLEMENTALOILANDGASDISCLOSU...,True,{'error_tags': {'tag': 'us-gaap:CostsIncurredA...
4,DQC.US.0015.576,"2022-02-23 06:12:12,070 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,1841666,APA Corp,2022-02-22 14:30:00,0001784031-22-000009,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:CostsIncurredAcquisitionOfOil...,"{'extracted_value': '-157,000,000', 'calculate...",10k-apa-20211231/SUPPLEMENTALOILANDGASDISCLOSU...,True,{'error_tags': {'tag': 'us-gaap:CostsIncurredA...
5,DQC.US.0015.576,"2022-02-23 06:12:59,857 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,6769,APACHE CORPORATION,2022-02-22 14:34:00,0001784031-22-000010,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,{'tag': 'us-gaap:CostsIncurredAcquisitionOfOil...,"{'extracted_value': '-157000000', 'calculated_...",10k-apa-20211231/SUPPLEMENTALOILANDGASDISCLOSU...,True,{'error_tags': {'tag': 'us-gaap:CostsIncurredA...
6,DQC.US.0015.576,"2022-02-23 06:12:59,858 [DQC.US.0015.576] us-g...",2022-02-23,ERROR,DQC,15,6769,APACHE CORPORATION,2022-02-22 14:34:00,0001784031-22-000010,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xm

In [24]:
new_mr_15_df.to_excel(output_data,index=False)